In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

api_key = os.environ["OPENAI_API_KEY"]
print("API key loaded successfully")

## A. Define DOI and STAC links

In [ ]:
stac_url: str = "https://earth.gov/ghgcenter/api/stac"
stac_collection_id: str = "vulcan-ffco2-yeargrid-v4"
publication_doi: str = "https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2020JD032974"

In [ ]:
from helper import download_stac_data

[collection_file_path, items_file_path, collection_data, items] = download_stac_data(stac_url=stac_url, stac_collection_id=stac_collection_id, output_dir="./data")


## B. Then download the stac collection and items

## 1. Data scraper to scrapes out the data from literature pdfs.

In [ ]:
from scraper import get_default_scraper

default_scraper = get_default_scraper(debug=False)

scraper_input = default_scraper.input_schema(
  url = "https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2020JD032974"
)

scraped_output = await default_scraper.arun(scraper_input)
scraped_text = scraped_output.content

In [ ]:
scraped_text

## 2. Data Format from the Data Search Agent

In [ ]:
import json
from helper import parse_stac_items_to_collection_items
from data_types import SearchedSTACData, CollectionItem
from typing import List

# with open("./data/stac_collection.json", "r") as f:
#   stac_collection = json.load(f)

# with open("./data/stac_items.json", "r") as f:
#   stac_collection_items = json.load(f)

stac_collection = collection_data
stac_collection_items = items

collection_items: List[CollectionItem] = parse_stac_items_to_collection_items(stac_collection_items, stac_collection)

In [ ]:
collection_items

## 3. Data Scount Agent 

### a. Relevant data filter component

In [ ]:
# from relevant_data_filter_agent import get_relevant_data

# relevant_data: List[CollectionItem] = await get_relevant_data(api_key=api_key, literature_context=scraped_text, stac_data=collection_items)

In [ ]:
# print("number of collection_items: ", len(collection_items), "\nnumber of relevant_data: ", len(relevant_data))

In [ ]:
# json_relevant_data = [rd.model_dump_json() for rd in relevant_data]
# print(json_relevant_data)

The above are commented as these subagents are part of the Data Scout Agent already.

### b. Data relationship builder component

In [ ]:
#NA

This subagent is part of the Data Scout Agent already. 

#### DATA SCOUT AGENT IMPLEMENTATION

In [ ]:
from data_scout_agent import DataScoutAgent, DataScoutAgentInputSchema, DataScoutAgentOutputSchema, DataScoutAgentConfig

data_scout_input: DataScoutAgentInputSchema = DataScoutAgentInputSchema(
  literature=scraped_text,
  collection_items=collection_items
)
data_scout_agent_config: DataScoutAgentConfig = DataScoutAgentConfig(api_key=api_key)
data_scout_agent: DataScoutAgent = DataScoutAgent(config=data_scout_agent_config)
data_scout_agent_output: DataScoutAgentOutputSchema = await data_scout_agent.arun(data_scout_input)

In [ ]:
data_scout_agent_output

In [ ]:
print("Number of input collections items: ", len(data_scout_input.collection_items))
print("Number of relevant collections items: ", len(data_scout_agent_output.relevant_collection))

## 4. Script writer agent

### a. Script blueprint builder component

In [ ]:
from script_blueprint_builder_agent import ScriptBlueprintBuilderAgent, ScriptBlueprintBuilderAgentConfig, ScriptBlueprintBuilderAgentInputSchema, ScriptBlueprintBuilderAgentOutputSchema

config = ScriptBlueprintBuilderAgentConfig(api_key=api_key)

script_blueprint_input: ScriptBlueprintBuilderAgentInputSchema = ScriptBlueprintBuilderAgentInputSchema(
  literature_text=scraped_text,
  collection_items=data_scout_agent_output.relevant_collection
)

script_blueprint_agent: ScriptBlueprintBuilderAgent = ScriptBlueprintBuilderAgent(config=config)

script_blueprint_output: ScriptBlueprintBuilderAgentOutputSchema = await script_blueprint_agent.arun(script_blueprint_input)

In [ ]:
script_blueprint_output.script_blueprint

### b. Script builder component

In [ ]:
from script_builder_agent import ScriptBuilderAgent, ScriptBuilderAgentConfig, ScriptBuilderAgentInputSchema, ScriptBuilderAgentOutputSchema

script_builder_config = ScriptBuilderAgentConfig(api_key=api_key)
script_builder_input = ScriptBuilderAgentInputSchema(
  narrative_blueprint=script_blueprint_output.script_blueprint
)
script_builder_agent: ScriptBuilderAgent = ScriptBuilderAgent(config=script_builder_config)
script_builder_output: ScriptBuilderAgentOutputSchema = await script_builder_agent.arun(script_builder_input)

In [ ]:
script_builder_output.story_draft

### c. Script Critique

In [ ]:
## Lets assume that we do not need critique.
## TODO: make this after data injection component

#### SCRIPT WRITER IMPLEMENTATION

In [ ]:
# from script_writer_agent import ScriptWriterAgent, ScriptWriterAgentConfig, ScriptWriterAgentInputSchema, ScriptWriterAgentOutputSchema

# script_writer_config = ScriptWriterAgentConfig(api_key=api_key)

# script_writer_input: ScriptWriterAgentInputSchema = ScriptWriterAgentInputSchema(
#   literature_text=scraped_text,
#   collection_items=data_scout_agent_output.relevant_collection
# )
# script_writer_agent: ScriptWriterAgent = ScriptWriterAgent(config=script_writer_config)
# script_writer_output: ScriptWriterAgentOutputSchema = await script_writer_agent.arun(script_writer_input)

In [ ]:
# script_writer_output.script

## 5. Data Injection Component

In [ ]:
from data_injection_agent import DataInjectionAgent, DataInjectionAgentConfig, DataInjectionAgentInputSchema, DataInjectionAgentOutputSchema

data_injection_agent_config: DataInjectionAgentConfig = DataInjectionAgentConfig(api_key=api_key)

data_injection_agent_input: DataInjectionAgentInputSchema = DataInjectionAgentInputSchema(
  script=script_builder_output.story_draft,
  collection_items=data_scout_agent_output.relevant_collection
)

data_injection_agent: DataInjectionAgent = DataInjectionAgent(config=data_injection_agent_config)
data_injected_script: DataInjectionAgentOutputSchema = await data_injection_agent.arun(data_injection_agent_input)


In [ ]:
data_injected_script.script_with_data

## 6. MDX Builder Component

In [ ]:
from mdx_builder_agent import MDXBuilderAgent, MDXBuilderAgentConfig, MDXBuilderAgentInputSchema, MDXBuilderAgentOutputSchema

mdx_builder_config: MDXBuilderAgentConfig = MDXBuilderAgentConfig(api_key=api_key)

mdx_builder_input_schema: MDXBuilderAgentInputSchema = MDXBuilderAgentInputSchema(
    story_script=data_injected_script.script_with_data
  )

mdx_builder_agent: MDXBuilderAgent = MDXBuilderAgent(mdx_builder_config)
mdx_builder_output: MDXBuilderAgentOutputSchema = await mdx_builder_agent.arun(mdx_builder_input_schema)

mdx_story: str = mdx_builder_output.story_mdx

In [ ]:
mdx_story

Test the MDX block validator

In [ ]:

mdx_story = """
  '<Block>\n  <Prose>\n    # Tracking CO₂: Understanding Emission Contributions Across the U.S.\n\n    ## Introduction\n\n    Our journey begins in the heart of America, specifically in Chautauqua County, Kansas. Here, using precise coordinates found at *[-96,37]*, we aim to unravel the intricate web of carbon dioxide emissions from various sectors.\n    Imagine carbon emissions gradually spiraling from different industrial activities, commercial sectors, and city outskirts, overlaying a map highlighting Kansas.\n    Understanding CO₂ emissions across the United States is essential to addressing climate change. The **Vulcan Project** and its latest dataset, version 4.0, bring to light high-resolution annual estimates detailing emissions from fossil fuels and cement production, illuminating the path ahead for policy and science.\n  </Prose>\n</Block>\n\n<Block>\n  <Prose>\n    ## The Data Event: Vulcan FFCO₂ v4.0\n\n    Between 2010 and 2021, the Vulcan Project collected and analyzed data down to a **1 km grid**.\n    Visualize an animated timeline of increasing emission data from **2010 to 2015** over the U.S. map, culminating in a detailed presentation of CO₂ emissions in **2015** in Chautauqua County.\n    Each year, the Vulcan Project compiles emissions data, enabling visualization of comprehensive carbon footprints. This tapestry of data presents a granular view of emissions coming from airports, marine vessels, railroads, and more, revealing how energy needs and industrial actions contribute to the emissions landscape.\n  </Prose>\n  <Figure>\n    <Map\n      datasetId="vulcan-ffco2-yeargrid-v4"\n      layerId="vulcan-ffco2-yeargrid-v4-2015"\n      dateTime="2015-01-01"\n    />\n    <Caption>\n      Visualization of CO₂ emissions in 2015 in Chautauqua County.\n    </Caption>\n  </Figure>\n</Block>\n\n<Block>\n  <Prose>\n    ## Scientific Validation and Assurance\n\n    Utilizing unique item identifiers, such as *vulcan-ffco2-yeargrid-v4-2015*, lends specificity and validation within scientific and policy discussions.\n    Display key data points, graphs illustrating emission trends, and confidence intervals signifying scientific rigor.\n    The Vulcan set acknowledges scientific and methodology advancements, enhancing our understanding of uncertainty and variability at finer scales than ever before. Beyond aggregating data, Vulcan v4.0 affirms its utility by offering a blueprint to cities seeking comprehensive inventories.\n  </Prose>\n</Block>\n\n<Block>\n  <Prose>\n    ## Mapping the Future\n\n    2015 marks our chosen benchmark year, where emissions patterns become crucial signs for future planning.\n    Imagine a futuristic cityscape shifting between excessive emissions and optimized, sustainable practices inspired by data.\n    With emissions from 2015 serving as our guiding baseline, we can juxtapose paths we\'ve tread with the sustainable futures we aim to achieve. The role of emissions monitoring is pivotal not only for its scientific merit but also for drafting actionable climate policies.\n  </Prose>\n</Block>\n\n<Block>\n  <Prose>\n    ## Conclusion\n\n    A final glance over Nebraska and Kansas reflects wider challenges and opportunities in emission management.\n    Visualize overlay maps showing before and after scenarios based on theoretical CO₂ reductions.\n    Every ton of CO₂ we track is a step toward a cleaner future. The Vulcan Project stands as a cornerstone endeavor, transforming raw data into meaningful narratives that steer our global dialogue toward a sustainable future.\n  </Prose>\n</Block>'
"""

In [4]:
from helper import MDXValidator

validator = MDXValidator()
validator.validate_mdx(mdx_story)

False